In [1]:
# ============================================================
# Import Required Libraries
# ============================================================

import os
import re
import json
import torch
import torch.nn.functional as F

from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("=" * 60)
print("Libraries Imported Successfully")
print("=" * 60)

C:\Users\shyam\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries Imported Successfully


In [2]:
# ============================================================
# GPU Configuration
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("Device :", device)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 60)

Device : cuda
GPU : NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
# ============================================================
# Load Teacher Model
# ============================================================

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True

)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

teacher_model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    quantization_config=bnb_config,

    torch_dtype=torch.float16,

    device_map="auto"

)

teacher_model.eval()

print("=" * 60)
print("Teacher Model Loaded Successfully")
print("=" * 60)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W0730 15:18:35.673000 2836 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|███████████████████████████████████████████████████████████████| 434/434 [00:09<00:00, 47.21it/s]


Teacher Model Loaded Successfully


In [4]:
# ============================================================
# Teacher Evaluation Prompt
# ============================================================

teacher_prompt = """
You are a senior board-certified thoracic radiologist with over 25 years of experience.

Your ONLY task is to evaluate the quality of a student-generated radiology report by comparing it with the Ground Truth Report.

You are NOT generating a radiology report.

You are ONLY grading the student report.

===========================================================
EVALUATION GUIDELINES
===========================================================

Compare the Student Report with the Ground Truth Report.

Evaluate based on:

• Correct medical findings
• Missed important findings
• Hallucinated findings
• Clinical reasoning
• Final diagnosis

Ignore wording differences.

Ignore report style.

Ignore report length.

Focus ONLY on factual medical correctness.

Be strict and objective.

===========================================================
SCORING
===========================================================

Assign ONE final quality score between 0.0 and 10.0.

Scoring Guide:

10 = Nearly perfect report

8–9 = Very good with minor mistakes

6–7 = Mostly correct but several mistakes

4–5 = Moderate quality with important errors

2–3 = Poor report with many errors

0–1 = Completely incorrect or hallucinated report

You may use increments of 0.5 only.

===========================================================
OUTPUT FORMAT
===========================================================

Return ONLY ONE line.

Score: X.X/10

Do NOT explain.

Do NOT justify.

Do NOT summarize.

Do NOT output anything else.
"""

In [5]:
# ============================================================
# Teacher Evaluation Function
# ============================================================

import re

def evaluate_trajectory(ground_truth, trajectory):

    prompt = f"""
{teacher_prompt}

===========================================================

GROUND TRUTH REPORT

{ground_truth}

===========================================================

STUDENT REPORT

{trajectory}

===========================================================

Evaluate the student report.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(device)

    with torch.no_grad():

        outputs = teacher_model.generate(

            **inputs,

            max_new_tokens=40,

            do_sample=False,

            temperature=0.0,

            repetition_penalty=1.1,

            pad_token_id=tokenizer.eos_token_id,

            eos_token_id=tokenizer.eos_token_id

        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    response = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    # Extract only the score
    match = re.search(
        r"Score:\s*([0-9]+(?:\.[0-9]+)?)/10",
        response,
        re.IGNORECASE
    )

    if match:
        return float(match.group(1))

    return None

In [104]:
# ============================================================
# Load Student Trajectories
# ============================================================

import os
import json

TRAJECTORY_DIR = "generated_trajectories"

trajectory_files = sorted(
    [
        f for f in os.listdir(TRAJECTORY_DIR)
        if f.endswith(".json")
    ]
)

print(f"Found {len(trajectory_files)} trajectory files.")

Found 1 trajectory files.


In [105]:
# ============================================================
# Load One Sample
# ============================================================

sample_file = trajectory_files[0]

sample_path = os.path.join(
    TRAJECTORY_DIR,
    sample_file
)

with open(sample_path, "r") as f:
    sample = json.load(f)

ground_truth = sample["ground_truth"]

trajectories = sample["trajectories"]

print("Sample:", sample_file)
print("Ground Truth Loaded")
print("Number of Trajectories:", len(trajectories))

Sample: sample_00000485kjabfjafjahfujashf.json
Ground Truth Loaded
Number of Trajectories: 5


In [106]:
# ============================================================
# Preview Data
# ============================================================

print("=" * 80)
print("GROUND TRUTH")
print("=" * 80)
print(ground_truth)

print("\n")

for i, traj in enumerate(trajectories):

    print("=" * 80)
    print(f"Trajectory {i+1}")
    print("=" * 80)
    print(traj)
    print()

GROUND TRUTH
The lungs are moderately well inflated. Mild prominence of lung vasculature without frank pulmonary edema. No pleural effusions. Mild cardiomegaly as before. The patient is post extubation and removal of enteric tube. EKG leads overlie the chest wall. Multiple subacute to chronic fractures involving the right posterior fourth through eighth ribs noted. EKG leads overlie the chest wall.  Post extubation and removal of enteric tube. Mild prominence of lung vasculature without frank pulmonary edema.


Trajectory 1
Findings:
- The image shows a frontal chest X-ray of a patient with a portable X-ray machine setting. 

Medical devices and support equipment:
- There is an endotracheal tube placed in the trachea.
- There is a central venous catheter (CVC) placed in the right subclavian vein.
- A nasogastric tube is present.

Heart size and mediastinum:
- The heart size appears within normal limits.

Lungs:
- There is no evidence of significant consolidation, atelectasis, or lung o

In [107]:
# ============================================================
# Evaluate All Student Trajectories
# ============================================================

teacher_outputs = []

print("=" * 60)
print("Teacher Evaluation Started")
print("=" * 60)

for i, trajectory in enumerate(trajectories):

    print(f"\nEvaluating Trajectory {i+1}...")

    result = evaluate_trajectory(
        ground_truth,
        trajectory
    )

    teacher_outputs.append(result)

    print(f"Score: {result}/10")

print("\n" + "=" * 60)
print("Teacher Evaluation Completed")
print("=" * 60)

Teacher Evaluation Started

Evaluating Trajectory 1...
Score: 0.0/10

Evaluating Trajectory 2...
Score: 0.0/10

Evaluating Trajectory 3...
Score: 2.0/10

Evaluating Trajectory 4...
Score: 2.0/10

Evaluating Trajectory 5...
Score: 2.0/10

Teacher Evaluation Completed
